## Kerakli kutubxonalar

`torch` — model va tensorlar uchun, `requests` bilan `zipfile` — datasetni yuklab olish uchun,
`matplotlib` — grafiklar uchun.

In [1]:
import re
import time
import zipfile
from collections import Counter
from pathlib import Path
import requests

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Qurilmani tanlash

GPU mavjud bo'lsa (NVIDIA uchun CUDA, Apple Silicon uchun MPS) o'qitish ancha tez ketadi,
aks holda CPU ishlatiladi.

In [2]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
DEVICE = torch.device("cpu")
print('Device:', DEVICE)

Device: cpu


## Giperparametrlar

`WINDOW` — markaziy so'z atrofidagi kontekst oynasining kengligi.
`NEGATIVES` — har bir haqiqiy juftlikka nechta salbiy namuna olinishi.
`EMBED_DIM` — har bir so'z vektorining o'lchami.

In [3]:
MIN_COUNT = 10            # words rarer than this are dropped from the vocabulary
SUBSAMPLE_T = 1e-4        # threshold for discarding very frequent words
WINDOW = 5                # max distance between center and context word
NEGATIVES = 5             # negative samples per positive pair
NOISE_POWER = 0.75        # unigram distribution is raised to this power

EMBED_DIM = 100
BATCH_SIZE = 8192
BLOCK_SIZE = 1_000_000    # corpus chunk used to build pairs at a time

LEARNING_RATE = 0.003
EPOCHS = 5

torch.manual_seed(42)
generator = torch.Generator().manual_seed(42)

## text8 dataseti

text8 — Vikipediyaning birinchi 100 MB matni. U allaqachon tozalangan: faqat kichik harflar
va bo'shliqlar, tinish belgilari va markup yo'q. Shuning uchun tokenizatsiya oddiy `split()` bilan bajariladi.

In [4]:
URL = "http://mattmahoney.net/dc/text8.zip"

ZIP_PATH = Path("text8.zip")

if not ZIP_PATH.exists():
    print("Downloading dataset (31 MB)...")
    response = requests.get(URL, timeout=300)
    response.raise_for_status()
    ZIP_PATH.write_bytes(response.content)

with zipfile.ZipFile(ZIP_PATH) as archive:
    text = archive.read("text8").decode("utf-8")

# text8 is the first 100 MB of wikipedia, already lowercased and stripped
# of punctuation and markup, so tokenising is just a split on whitespace
tokens = text.split()

print("Characters:", len(text))
print("Tokens:", len(tokens))
print("Unique tokens:", len(set(tokens)))
print("\nFirst 25 tokens:", tokens[:25])

Characters: 100000000
Tokens: 17005207
Unique tokens: 253854

First 25 tokens: ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including', 'the', 'diggers', 'of', 'the', 'english', 'revolution', 'and', 'the', 'sans', 'culottes']


## Lug'at tuzish

Har bir so'z nechta marta uchraganini sanaymiz va `MIN_COUNT` dan kam uchraganlarini tashlaymiz —
bir necha marta uchragan so'z uchun yaxshi vektor o'rganib bo'lmaydi.

In [5]:
def build_vocab(tokens, min_count):
    counter = Counter(tokens)
    kept = [(w, c) for w, c in counter.most_common() if c >= min_count]

    itos = [w for w, _ in kept]
    stoi = {w: i for i, w in enumerate(itos)}
    counts = torch.tensor([c for _, c in kept], dtype=torch.float)

    return stoi, itos, counts


stoi, itos, counts = build_vocab(tokens, MIN_COUNT)
vocab_size = len(itos)

coverage = counts.sum().item() / len(tokens)

print("Vocabulary size:", vocab_size)
print(f"Corpus coverage: {coverage:.2%}")
print("\nMost frequent:", [(itos[i], int(counts[i])) for i in range(10)])
print("Least frequent:", [(itos[i], int(counts[i])) for i in range(vocab_size - 5, vocab_size)])

Vocabulary size: 47134
Corpus coverage: 97.39%

Most frequent: [('the', 1061396), ('of', 593677), ('and', 416629), ('one', 411764), ('in', 372201), ('a', 325873), ('to', 316376), ('zero', 264975), ('nine', 250430), ('two', 192644)]
Least frequent: [('montrealers', 10), ('cephalon', 10), ('meherabad', 10), ('villein', 10), ('kirchenmusik', 10)]


## So'zlarni indekslarga aylantirish

Model matn bilan emas, raqamlar bilan ishlaydi. Har bir so'z o'z indeksiga almashtiriladi.

In [6]:
def encode_tokens(tokens, stoi):
    return torch.tensor([stoi[w] for w in tokens if w in stoi], dtype=torch.long)


ids = encode_tokens(tokens, stoi)

print("Encoded corpus:", tuple(ids.shape))
print("Dropped as out-of-vocabulary:", len(tokens) - len(ids))
print("\nFirst 15 ids  :", ids[:15].tolist())
print("decoded back  :", [itos[i] for i in ids[:15].tolist()])

Encoded corpus: (16561031,)
Dropped as out-of-vocabulary: 444176

First 15 ids  : [5233, 3080, 11, 5, 194, 1, 3133, 45, 58, 155, 127, 741, 476, 10571, 133]
decoded back  : ['anarchism', 'originated', 'as', 'a', 'term', 'of', 'abuse', 'first', 'used', 'against', 'early', 'working', 'class', 'radicals', 'including']


## Tez-tez uchraydigan so'zlarni kamaytirish (subsampling)

"the", "of" kabi so'zlar o'z qo'shnisi haqida deyarli hech qanday ma'lumot bermaydi.
word2vec ularning ko'p qismini tasodifiy tashlab yuboradi — bu ham tezlikni oshiradi,
ham vektorlar sifatini yaxshilaydi, chunki kam uchraydigan so'zlar atrofidagi oyna kengayadi.

In [7]:
def subsample(ids, counts, threshold, generator):
    """Randomly discard frequent words: P(keep) = (sqrt(f/t) + 1) * (t/f).

    'the' carries almost no information about its neighbours, so word2vec
    throws most occurrences away. This speeds training up AND improves the
    vectors, because it effectively widens the window around rare words.
    """
    freq = counts / counts.sum()
    keep_prob = ((freq / threshold).sqrt() + 1) * (threshold / freq)
    keep_prob = keep_prob.clamp(max=1.0)

    draws = torch.rand(len(ids), generator=generator)
    mask = draws < keep_prob[ids]

    return ids[mask], keep_prob


train_ids, keep_prob = subsample(ids, counts, SUBSAMPLE_T, generator)

print("Tokens before subsampling:", len(ids))
print("Tokens after subsampling :", len(train_ids))
print(f"Kept: {len(train_ids) / len(ids):.1%}")

print("\nKeep probability per word:")
for w in ["the", "of", "and", "one", "history", "king", "physics", "chess"]:
    if w in stoi:
        i = stoi[w]
        print(f"  {w:<10} count={int(counts[i]):>8}  P(keep)={keep_prob[i]:.4f}")

Tokens before subsampling: 16561031
Tokens after subsampling : 9214213
Kept: 55.6%

Keep probability per word:
  the        count= 1061396  P(keep)=0.0411
  of         count=  593677  P(keep)=0.0556
  and        count=  416629  P(keep)=0.0670
  one        count=  411764  P(keep)=0.0674
  history    count=   12623  P(keep)=0.4934
  king       count=    7456  P(keep)=0.6934
  physics    count=    1492  P(keep)=1.0000
  chess      count=     616  P(keep)=1.0000


## Shovqin taqsimoti (3/4 darajasi)

Salbiy namunalar so'zlar chastotasidan olinadi, lekin chastota 0.75 darajaga ko'tariladi.
Bu taqsimotni tekislaydi: kam uchraydigan so'zlar ko'proq, ko'p uchraydiganlari kamroq tanlanadi.
Bu word2vec maqolasidagi sof empirik usul, ammo sifatga katta ta'sir qiladi.

In [8]:
# Negative samples are drawn from the unigram distribution raised to 3/4.
# That power flattens the distribution: rare words get sampled more often
# than their raw frequency, common words less. It is a purely empirical
# trick from the word2vec paper, but it matters a lot for quality.
noise_dist = counts.pow(NOISE_POWER)
noise_dist = noise_dist / noise_dist.sum()

unigram = counts / counts.sum()

print(f"{'word':<12}{'P(unigram)':>14}{'P(^0.75)':>14}{'ratio':>10}")
for w in ["the", "of", "one", "history", "king", "physics", "chess"]:
    if w in stoi:
        i = stoi[w]
        ratio = (noise_dist[i] / unigram[i]).item()
        print(f"{w:<12}{unigram[i]:>14.6f}{noise_dist[i]:>14.6f}{ratio:>10.2f}x")

word            P(unigram)      P(^0.75)     ratio
the               0.064090      0.015435      0.24x
of                0.035848      0.009983      0.28x
one               0.024863      0.007587      0.31x
history           0.000762      0.000556      0.73x
king              0.000450      0.000375      0.83x
physics           0.000090      0.000112      1.24x
chess             0.000037      0.000058      1.55x


## Salbiy namunalarni tanlash

Teskari CDF usuli: tasodifiy son olamiz va u kumulyativ taqsimotda qayerga tushganini
`searchsorted` bilan topamiz. Bitta chaqiruv butun batch uchun namuna beradi.

In [9]:
# Sampling by inverse CDF: draw u ~ U(0,1) and find where it lands in the
# cumulative distribution. One searchsorted call gives a whole batch.
noise_cdf = noise_dist.cumsum(0).to(DEVICE)


def sample_negatives(n_rows, n_negatives):
    draws = torch.rand(n_rows, n_negatives, device=DEVICE)
    return torch.searchsorted(noise_cdf, draws).clamp(max=vocab_size - 1)


sample = sample_negatives(3, NEGATIVES)
print("negative sample ids:\n", sample)
print("\nas words:")
for row in sample.tolist():
    print(" ", [itos[i] for i in row])

negative sample ids:
 tensor([[20191, 25006,   945, 34182,  1004],
        [ 3976,   268, 11849, 29879,    32],
        [28604,  3811, 18626,  3275,  8782]])

as words:
  ['gchq', 'declination', 'congress', 'mollusks', 'units']
  ['baltimore', 'line', 'katakana', 'pumpkins', 'this']
  ['keiretsu', 'rotation', 'idempotent', 'humanity', 'turkic']


## (markaz, kontekst) juftliklarini hosil qilish

Barcha juftliklarni bir vaqtda xotiraga sig'dirib bo'lmaydi — bir necha GB kerak bo'lardi.
Shuning uchun korpus bloklarga bo'lib o'tiladi va har bir siljish uchun butun blok
tensor sifatida siljitiladi. So'zlar bo'yicha Python sikli yo'q.

In [10]:
def skipgram_pairs(ids, window, block_size, generator):
    """Yield (center, context) id tensors, one shuffled block at a time.

    Materialising every pair at once would need several GB, so the corpus is
    walked in blocks. For each offset d the whole block is shifted by d and
    paired elementwise - no Python loop over words.
    """
    for start in range(0, len(ids), block_size):
        block = ids[start:start + block_size]
        if len(block) <= window:
            continue

        centers, contexts = [], []

        for d in range(1, window + 1):
            # word2vec shrinks the window at random for every center word;
            # equivalently, an offset d survives with prob (window-d+1)/window,
            # so nearby words are sampled more often than distant ones
            keep = (window - d + 1) / window

            left, right = block[:-d], block[d:]

            mask = torch.rand(len(left), generator=generator) < keep
            centers.append(left[mask])
            contexts.append(right[mask])

            mask = torch.rand(len(left), generator=generator) < keep
            centers.append(right[mask])
            contexts.append(left[mask])

        centers = torch.cat(centers)
        contexts = torch.cat(contexts)

        perm = torch.randperm(len(centers), generator=generator)
        yield centers[perm], contexts[perm]

## Juftliklarni tekshirish

Hosil bo'lgan juftliklardan bir nechtasini so'zlarga qaytarib ko'ramiz.

In [11]:
peek_centers, peek_contexts = next(skipgram_pairs(train_ids, WINDOW, 200_000, generator))

print("pairs from one block:", len(peek_centers))
print("estimated pairs per epoch:", round(len(peek_centers) * len(train_ids) / 200_000))

print("\nfirst 10 training pairs:")
for c, x in zip(peek_centers[:10].tolist(), peek_contexts[:10].tolist()):
    print(f"  {itos[c]:<15} -> {itos[x]}")

pairs from one block: 1201161
estimated pairs per epoch: 55338767

first 10 training pairs:
  jim             -> thinking
  four            -> manner
  aristotle       -> plato
  cut             -> marcian
  line            -> arabic
  archaeologists  -> processing
  more            -> limits
  stated          -> publicly
  babies          -> fifty
  fog             -> currents


## Model: Skip-gram + Negative Sampling

Darsda ko'rilgan loss funksiyasining o'zi. Ikkita alohida embedding matritsasi bor:
`in_embed` (markaziy so'zlar) va `out_embed` (kontekst so'zlari). Oxirida faqat birinchisi ishlatiladi.

In [12]:
class SkipGramNegativeSampling(nn.Module):
    """The loss from the lesson:

        -log sigma(v_c . u_o)  -  sum_k log sigma(-v_c . u_k)

    v comes from the input matrix, u from the output matrix. Two separate
    embedding tables for the same words - only the input one is kept at the end.
    """

    def __init__(self, vocab_size, embed_dim):
        super().__init__()

        self.in_embed = nn.Embedding(vocab_size, embed_dim)
        self.out_embed = nn.Embedding(vocab_size, embed_dim)

        # word2vec's initialisation: small random inputs, zeroed outputs
        nn.init.uniform_(self.in_embed.weight, -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.zeros_(self.out_embed.weight)

    def forward(self, centers, contexts, negatives):
        v = self.in_embed(centers)                          # [B, D]
        u_pos = self.out_embed(contexts)                    # [B, D]
        u_neg = self.out_embed(negatives)                   # [B, K, D]

        pos_score = (v * u_pos).sum(dim=1)                  # [B]
        neg_score = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)   # [B, K]

        pos_loss = F.logsigmoid(pos_score)                  # push pairs together
        neg_loss = F.logsigmoid(-neg_score).sum(dim=1)      # push samples apart

        return -(pos_loss + neg_loss).mean()

## Modelni yaratish

`out_embed` nollar bilan boshlanadi, shuning uchun barcha skalyar ko'paytmalar 0 ga teng bo'ladi
va birinchi loss aniq `-(1+K)·log(0.5)` chiqishi kerak. Bu — kodni tekshirishning oson yo'li.

In [13]:
model = SkipGramNegativeSampling(vocab_size, EMBED_DIM).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())

print(model)
print(total_params, "parameters")

# with out_embed at zero every score starts at 0, so the first loss should be
# -(1 + NEGATIVES) * log(0.5)
print("\nexpected initial loss:", round(-(1 + NEGATIVES) * torch.tensor(0.5).log().item(), 4))

SkipGramNegativeSampling(
  (in_embed): Embedding(47134, 100)
  (out_embed): Embedding(47134, 100)
)
9426800 parameters

expected initial loss: 4.1589


## Optimizator

In [14]:
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

## Bitta epoxani o'qitish

Har bir batch uchun: juftliklarni olamiz, salbiy namunalarni tanlaymiz,
loss hisoblaymiz va gradientlarni yangilaymiz.

In [15]:
def train_one_epoch(model, optimizer, ids):
    model.train()

    total_loss = torch.zeros((), device=DEVICE)
    n_batches = 0

    for centers, contexts in skipgram_pairs(ids, WINDOW, BLOCK_SIZE, generator):
        for i in range(0, len(centers) - BATCH_SIZE + 1, BATCH_SIZE):
            c = centers[i:i + BATCH_SIZE].to(DEVICE)
            x = contexts[i:i + BATCH_SIZE].to(DEVICE)
            negatives = sample_negatives(len(c), NEGATIVES)

            optimizer.zero_grad()
            loss = model(c, x, negatives)
            loss.backward()
            optimizer.step()

            total_loss += loss.detach()      # kept on device to avoid syncing
            n_batches += 1

    return (total_loss / n_batches).item()

## O'qitishni boshlash

Bu notebookdagi eng uzoq qadam.

In [17]:
losses = []

for epoch in range(1, EPOCHS + 1):
    started = time.time()
    loss = train_one_epoch(model, optimizer, train_ids)
    losses.append(loss)

    print(
        f"Epoch [{epoch}/{EPOCHS}] "
        f"| Loss: {loss:.4f} "
        f"| {time.time() - started:.0f}s"
    )

KeyboardInterrupt: 

## Loss grafigi

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(1, EPOCHS + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Negative sampling loss")
plt.title("Skip-gram with negative sampling - text8")
plt.grid(True)
plt.show()

## Vektorlar va o'xshashlik

Faqat `in_embed` matritsasi ishlatiladi. Vektorlarni normallashtiramiz — shunda skalyar
ko'paytma to'g'ridan-to'g'ri kosinus o'xshashligiga teng bo'ladi va har bir so'rov bitta matmul.

In [ ]:
# only the input matrix is used. Normalising the rows makes a dot product
# equal to the cosine similarity, so every query below is one matmul.
vectors = model.in_embed.weight.detach().cpu()
vectors = vectors / vectors.norm(dim=1, keepdim=True).clamp(min=1e-9)

print("Embedding matrix:", tuple(vectors.shape))


def most_similar(word, k=8, vectors=vectors, stoi=stoi, itos=itos):
    if word not in stoi:
        return f"'{word}' is not in the vocabulary"

    scores = vectors @ vectors[stoi[word]]
    scores[stoi[word]] = -2.0                # never return the query itself

    top = scores.topk(k)
    return [(itos[i], round(s, 3)) for i, s in zip(top.indices.tolist(), top.values.tolist())]


def similarity(a, b, vectors=vectors, stoi=stoi):
    if a not in stoi or b not in stoi:
        return float("nan")
    return (vectors[stoi[a]] @ vectors[stoi[b]]).item()

## Eng yaqin so'zlar

Har bir so'z uchun vektor fazosida unga eng yaqin turgan so'zlar.

In [ ]:
for word in ["king", "paris", "computer", "water", "war", "music", "three"]:
    print(f"{word}:")
    for neighbour, score in most_similar(word, k=6):
        print(f"    {neighbour:<16} {score:.3f}")
    print()

## Sinonimlar va bog'liq bo'lmagan so'zlar

Bitta kosinus qiymati o'z-o'zidan hech narsani anglatmaydi. Uni tasodifiy tanlangan
so'zlar juftligining o'xshashligi bilan solishtirish kerak — asosiy ma'no shu ikkisi orasidagi farqda.

In [ ]:
# The number on its own means nothing - it only means something next to the
# similarity of two words picked at random, which is what "unrelated" looks like.
pairs_related = [
    ("car", "automobile"), ("good", "great"), ("happy", "joyful"),
    ("king", "queen"), ("france", "germany"), ("dog", "cat"),
    ("doctor", "physician"), ("big", "large"),
]

pairs_unrelated = [
    ("car", "banana"), ("good", "uranium"), ("happy", "concrete"),
    ("king", "algebra"), ("france", "molecule"), ("dog", "november"),
]

torch.manual_seed(0)
frequent = torch.randint(0, 5000, (20_000, 2))
frequent = frequent[frequent[:, 0] != frequent[:, 1]]
random_sims = (vectors[frequent[:, 0]] * vectors[frequent[:, 1]]).sum(dim=1)

print(f"{'pair':<28}{'cosine':>10}")
print("-" * 38)
for a, b in pairs_related:
    print(f"{a + ' / ' + b:<28}{similarity(a, b):>10.3f}")
print("-" * 38)
for a, b in pairs_unrelated:
    print(f"{a + ' / ' + b:<28}{similarity(a, b):>10.3f}")
print("-" * 38)
print(f"{'random pair (mean)':<28}{random_sims.mean():>10.3f}")
print(f"{'random pair (std)':<28}{random_sims.std():>10.3f}")

## Analogiyalar

`king - man + woman ≈ queen`. Uchta kirish so'zi natijadan chiqarib tashlanadi,
aks holda model ko'pincha ularning o'zini qaytaradi va analogiya ko'rinmay qoladi.

In [ ]:
def analogy(a, b, c, k=5, vectors=vectors, stoi=stoi, itos=itos):
    """a is to b as c is to ?   ->   b - a + c"""
    for w in (a, b, c):
        if w not in stoi:
            return f"'{w}' is not in the vocabulary"

    target = vectors[stoi[b]] - vectors[stoi[a]] + vectors[stoi[c]]
    target = target / target.norm().clamp(min=1e-9)

    scores = vectors @ target
    # the three inputs sit closest to the result almost every time; excluding
    # them is what makes the analogy visible at all
    for w in (a, b, c):
        scores[stoi[w]] = -2.0

    top = scores.topk(k)
    return [(itos[i], round(s, 3)) for i, s in zip(top.indices.tolist(), top.values.tolist())]


questions = [
    ("man", "king", "woman"),
    ("france", "paris", "italy"),
    ("walk", "walking", "swim"),
    ("big", "bigger", "small"),
    ("good", "better", "bad"),
    ("english", "england", "french"),
]

for a, b, c in questions:
    print(f"{a} : {b}  ::  {c} : ?")
    print("   ", analogy(a, b, c, k=4))
    print()

## 2D ga proyeksiya (PCA)

Butun lug'atni chizsak tushunarsiz nuqtalar to'dasi chiqadi. Shuning uchun faqat
tanlab olingan so'z guruhlari ko'rsatiladi — o'shanda tuzilma aniq ko'rinadi.

In [ ]:
# A curated word list projected to 2D. Running PCA over the whole vocabulary
# gives an unreadable blob - the structure only shows up on chosen groups.
groups = {
    "countries": ["france", "germany", "italy", "spain", "japan", "china"],
    "capitals": ["paris", "berlin", "rome", "madrid", "tokyo", "beijing"],
    "numbers": ["one", "two", "three", "four", "five", "six"],
    "animals": ["dog", "cat", "horse", "cow", "sheep", "pig"],
    "verbs": ["run", "walk", "swim", "jump", "sing", "write"],
}

words = [w for group in groups.values() for w in group if w in stoi]
labels = [g for g, group in groups.items() for w in group if w in stoi]

X = vectors[[stoi[w] for w in words]]
X = X - X.mean(dim=0)

# PCA via SVD: the first two right singular vectors are the top components
_, _, V = torch.linalg.svd(X, full_matrices=False)
xy = X @ V[:2].T

plt.figure(figsize=(11, 8))
for group in groups:
    mask = [i for i, l in enumerate(labels) if l == group]
    plt.scatter(xy[mask, 0], xy[mask, 1], s=60, label=group)

for i, word in enumerate(words):
    plt.annotate(word, (xy[i, 0], xy[i, 1]), fontsize=9,
                 xytext=(4, 4), textcoords="offset points")

plt.title("Word vectors projected to 2D (PCA)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Ortiqcha so'zni topish

Guruhdagi so'zlarning o'rtacha vektoridan eng uzoq turgan so'z.

In [ ]:
def odd_one_out(words, vectors=vectors, stoi=stoi):
    known = [w for w in words if w in stoi]
    if len(known) < 3:
        return "not enough known words"

    X = vectors[[stoi[w] for w in known]]
    centroid = X.mean(dim=0)
    centroid = centroid / centroid.norm().clamp(min=1e-9)

    scores = X @ centroid
    return known[scores.argmin().item()]


tests = [
    ["breakfast", "lunch", "dinner", "uranium"],
    ["france", "germany", "italy", "guitar"],
    ["red", "blue", "green", "physics"],
    ["monday", "tuesday", "friday", "orange"],
]

for words_in in tests:
    print(f"{str(words_in):<52} -> {odd_one_out(words_in)}")

## Nega korpus katta bo'lishi kerak?

Xuddi shu kod va xuddi shu giperparametrlar, lekin 17 million o'rniga 200 ming so'z —
Shekspir matni. Farqni o'z ko'zi bilan ko'rish uchun.

In [ ]:
# Why the corpus had to be this big.
# Same code, same hyperparameters, 200k words of Shakespeare instead of 17M.
small_text = Path("tinyshakespeare.txt").read_text(encoding="utf-8").lower()
small_tokens = re.findall(r"[a-z']+", small_text)

small_stoi, small_itos, small_counts = build_vocab(small_tokens, min_count=2)
small_ids = encode_tokens(small_tokens, small_stoi)
small_ids, _ = subsample(small_ids, small_counts, SUBSAMPLE_T, generator)

print("Shakespeare tokens:", len(small_tokens), "| vocabulary:", len(small_itos))

small_model = SkipGramNegativeSampling(len(small_itos), EMBED_DIM).to(DEVICE)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=LEARNING_RATE)

small_cdf = (small_counts.pow(NOISE_POWER) / small_counts.pow(NOISE_POWER).sum()).cumsum(0).to(DEVICE)

_full_cdf, _full_vocab = noise_cdf, vocab_size
noise_cdf, vocab_size = small_cdf, len(small_itos)      # reuse sample_negatives

for epoch in range(1, 6):
    loss = train_one_epoch(small_model, small_optimizer, small_ids)
    print(f"  epoch {epoch} loss {loss:.4f}")

noise_cdf, vocab_size = _full_cdf, _full_vocab          # restore

small_vectors = small_model.in_embed.weight.detach().cpu()
small_vectors = small_vectors / small_vectors.norm(dim=1, keepdim=True).clamp(min=1e-9)

## Natijalarni solishtirish

Kichik korpusda sinonimlar bilan tasodifiy so'zlar orasidagi farq deyarli yo'qoladi.
Aynan shu farq — vektorlar haqiqatan ma'no o'rgangani yoki yo'qligining o'lchovi.

In [ ]:
print("nearest neighbours of 'king'\n")

print("  text8 (17M tokens):")
for w, s in most_similar("king", k=5):
    print(f"    {w:<16} {s:.3f}")

print("\n  shakespeare (0.2M tokens):")
for w, s in most_similar("king", k=5, vectors=small_vectors, stoi=small_stoi, itos=small_itos):
    print(f"    {w:<16} {s:.3f}")

# the honest measurement: how far related pairs sit above random pairs
def synonym_gap(vectors, stoi, pairs):
    known = [(a, b) for a, b in pairs if a in stoi and b in stoi]
    related = torch.tensor([(vectors[stoi[a]] @ vectors[stoi[b]]).item() for a, b in known])

    torch.manual_seed(0)
    n = min(2000, len(stoi))
    idx = torch.randint(0, n, (10_000, 2))
    idx = idx[idx[:, 0] != idx[:, 1]]
    baseline = (vectors[idx[:, 0]] * vectors[idx[:, 1]]).sum(dim=1)

    return related.mean().item(), baseline.mean().item(), len(known)


shared = [("king", "queen"), ("man", "woman"), ("day", "night"),
          ("good", "great"), ("love", "heart"), ("father", "mother")]

for name, v, s in [("text8", vectors, stoi), ("shakespeare", small_vectors, small_stoi)]:
    rel, base, n = synonym_gap(v, s, shared)
    print(f"\n{name}: related={rel:.3f}  random={base:.3f}  gap={rel - base:.3f}  ({n} pairs)")

## Saqlash

Vektorlar va lug'atni keyinchalik ishlatish uchun saqlaymiz.

In [ ]:
checkpoint_path = "word2vec_text8.pt"

torch.save(
    {
        "in_embed": model.in_embed.weight.detach().cpu(),
        "out_embed": model.out_embed.weight.detach().cpu(),
        "stoi": stoi,
        "itos": itos,
        "counts": counts,
        "embed_dim": EMBED_DIM,
        "window": WINDOW,
        "negatives": NEGATIVES,
        "min_count": MIN_COUNT,
    },
    checkpoint_path,
)

print("Saved to:", checkpoint_path)